# Part 2 companion — SQL vs pandas check (Task 2.2)
`02_queries.sql` holds the Part 2 queries. This notebook reproduces the clinic counts with pandas `groupby` and compares them with the same SQL query run on `load_data.sql` (loaded into an in-memory SQLite database, so it runs in Colab with no MySQL).

In [ ]:
import sqlite3
import pandas as pd

df = pd.read_csv("appointments_clean.csv")
clinics = pd.read_csv("clinics_enriched.csv")

# pandas side
pd_counts = (df.merge(clinics, on="clinic_id")
               .groupby(["clinic_name", "city"])
               .agg(appointments=("no_show", "size"), no_shows=("no_show", "sum"))
               .reset_index()
               .sort_values("clinic_name").reset_index(drop=True))
pd_counts["no_show_pct"] = (100 * pd_counts["no_shows"] / pd_counts["appointments"]).round(1)
print(pd_counts)

In [ ]:
# SQL side: build the schema, run load_data.sql, run the Task 2.2 query (without HAVING)
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE clinics (clinic_id VARCHAR(5) PRIMARY KEY, clinic_name VARCHAR(40) NOT NULL, city VARCHAR(20) NOT NULL,
  tier VARCHAR(10), latitude DECIMAL(8,4), longitude DECIMAL(8,4), temp_c DECIMAL(5,1));
CREATE TABLE appointments (appointment_id VARCHAR(8) PRIMARY KEY, patient_id VARCHAR(8) NOT NULL, clinic_id VARCHAR(5) NOT NULL,
  booked_on DATE, appt_date DATE, age INT, gender VARCHAR(1), distance_km DECIMAL(5,1), fee_inr INT,
  booking_channel VARCHAR(10), visit_type VARCHAR(12), reminder_sent VARCHAR(3), prior_no_shows INT,
  status VARCHAR(10), lead_days INT, no_show INT, FOREIGN KEY (clinic_id) REFERENCES clinics(clinic_id));
""")
con.executescript(open("load_data.sql").read())
sql_counts = pd.read_sql_query("""
SELECT c.clinic_name, c.city, COUNT(*) AS appointments, SUM(a.no_show) AS no_shows,
       ROUND(100.0 * SUM(a.no_show) / COUNT(*), 1) AS no_show_pct
FROM appointments a JOIN clinics c ON a.clinic_id = c.clinic_id
GROUP BY c.clinic_id, c.clinic_name, c.city
ORDER BY c.clinic_name""", con)
print(sql_counts)

same = pd_counts.equals(sql_counts.astype(pd_counts.dtypes.to_dict()))
print("pandas and SQL clinic counts / no-show counts / rates identical:", same)
assert (pd_counts[["appointments", "no_shows"]].values == sql_counts[["appointments", "no_shows"]].values).all()

**Result:** the pandas `groupby` counts match the SQL counts exactly for all 8 clinics. Without the `HAVING` filter, Gomti Nagar Care Point (Lucknow) has the highest rate (33.3%) but only 90 appointments, so its rate is less reliable than Hadapsar Care Centre's 27.3% on 143 appointments.